In [ ]:
!pip install torch-geometric


In [ ]:
import torch
import torch.nn.functional as F

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from torchvision.datasets import MNIST
from torchvision import transforms


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = MNIST(root='.', train=True, download=True, transform=transform)
test_dataset  = MNIST(root='.', train=False, download=True, transform=transform)


In [ ]:
def image_to_graph(image, label):
    # image is (1, H, W), squeeze to (H, W)
    image = image.squeeze(0)
    H, W = image.shape

    # Node features: pixel intensities
    x = image.view(-1, 1) # (H*W, 1)

    # Edge index: connect adjacent pixels (4-neighborhood)
    edge_indices = []
    for r in range(H):
        for c in range(W):
            idx = r * W + c
            # Connect to right neighbor
            if c + 1 < W:
                edge_indices.append([idx, r * W + (c + 1)])
                edge_indices.append([r * W + (c + 1), idx]) # Bidirectional
            # Connect to bottom neighbor
            if r + 1 < H:
                edge_indices.append([idx, (r + 1) * W + c])
                edge_indices.append([(r + 1) * W + c, idx]) # Bidirectional

    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()

    # Create Data object
    data = Data(x=x, edge_index=edge_index, y=label)
    return data

train_graphs = []
for i in range(2000):
    img, label = train_dataset[i]
    train_graphs.append(image_to_graph(img, torch.tensor(label)))

test_graphs = []
for i in range(1000):
    img, label = test_dataset[i]
    test_graphs.append(image_to_graph(img, torch.tensor(label)))

In [ ]:
print(train_graphs[67])


Data(x=[784, 1], edge_index=[2, 3024], y=1)


In [ ]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_graphs, batch_size=32)


In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(1, 64)
        self.conv2 = GCNConv(64, 128)
        self.fc = torch.nn.Linear(128, 10)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        x = global_mean_pool(x, batch)
        return self.fc(x)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
def train():
    model.train()
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = F.cross_entropy(out, data.y)
        loss.backward()
        optimizer.step()


In [ ]:
def test(loader):
    model.eval()
    correct = 0
    for data in loader:
        data = data.to(device)
        out = model(data)
        pred = out.argmax(dim=1)
        correct += int((pred == data.y).sum())
    return correct / len(loader.dataset)


In [ ]:
for epoch in range(1, 31):
    train()
    acc = test(test_loader)
    print(f"Epoch {epoch}, Test Accuracy: {acc:.4f}")


Epoch 1, Test Accuracy: 0.0890
Epoch 2, Test Accuracy: 0.1310
Epoch 3, Test Accuracy: 0.1840
Epoch 4, Test Accuracy: 0.2280
Epoch 5, Test Accuracy: 0.2130
Epoch 6, Test Accuracy: 0.2040
Epoch 7, Test Accuracy: 0.2150
Epoch 8, Test Accuracy: 0.2160
Epoch 9, Test Accuracy: 0.2190
Epoch 10, Test Accuracy: 0.2270
Epoch 11, Test Accuracy: 0.2310
Epoch 12, Test Accuracy: 0.2050
Epoch 13, Test Accuracy: 0.2090
Epoch 14, Test Accuracy: 0.2340
Epoch 15, Test Accuracy: 0.2160
Epoch 16, Test Accuracy: 0.2180
Epoch 17, Test Accuracy: 0.2080
Epoch 18, Test Accuracy: 0.2210
Epoch 19, Test Accuracy: 0.2260
Epoch 20, Test Accuracy: 0.2250
Epoch 21, Test Accuracy: 0.2310
Epoch 22, Test Accuracy: 0.2440
Epoch 23, Test Accuracy: 0.2090
Epoch 24, Test Accuracy: 0.2270
Epoch 25, Test Accuracy: 0.2410
Epoch 26, Test Accuracy: 0.2370
Epoch 27, Test Accuracy: 0.2360
Epoch 28, Test Accuracy: 0.2030
Epoch 29, Test Accuracy: 0.2240
Epoch 30, Test Accuracy: 0.2290
